<a href="https://colab.research.google.com/github/miaouiAmine/NLP-Translation-Demo/blob/main/chatBot_DistilGPT-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install required libraries
!pip install transformers torch datasets

# Step 2: Import necessary libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import random

# Step 3: Download and prepare dataset
def prepare_dataset():
    # Download Cornell Movie-Dialogs Corpus
    !wget https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
    !unzip cornell_movie_dialogs_corpus.zip

    # Process the dataset
    with open('cornell movie-dialogs corpus/movie_lines.txt', 'r', errors='ignore') as f:
        lines = f.readlines()

    # Create conversation pairs
    dialogues = {}
    for line in lines:
        parts = line.split(' +++$+++ ')
        dialogues[parts[0]] = parts[-1].strip()

    with open('cornell movie-dialogs corpus/movie_conversations.txt', 'r', errors='ignore') as f:
        conversations = f.readlines()

    pairs = []
    for conv in conversations:
        ids = eval(conv.split(' +++$+++ ')[-1])
        for i in range(len(ids)-1):
            pairs.append((dialogues[ids[i]], dialogues[ids[i+1]]))

    # Format data for training
    formatted_data = []
    for pair in pairs:
        formatted_data.append(f"<start> {pair[0]} <response> {pair[1]} <end>")

    # Split data
    random.shuffle(formatted_data)
    split = int(0.9 * len(formatted_data))
    train_data = formatted_data[:split]
    valid_data = formatted_data[split:]

    return train_data, valid_data

train_data, valid_data = prepare_dataset()

# Step 4: Initialize tokenizer and model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.add_tokens(['<start>', '<response>', '<end>'])

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# Step 5: Prepare dataset formatting
class ChatDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.data = data
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze()
        }

train_dataset = ChatDataset(train_data, tokenizer)
valid_dataset = ChatDataset(valid_data, tokenizer)

# Step 6: Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

# Step 7: Create Trainer and train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
)

trainer.train()

# Step 8: Save the model
model.save_pretrained('./chatbot_model')
tokenizer.save_pretrained('./chatbot_model')

# Step 9: Chat function
def chat():
    model.eval()
    print("Chat with the AI! Type 'quit' to exit.")

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            break

        prompt = f"<start> {user_input} <response>"
        inputs = tokenizer(prompt, return_tensors='pt')

        outputs = model.generate(
            inputs.input_ids,
            max_length=128,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.9,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        response = response.split('<response>')[-1].split('<end>')[0].strip()
        print(f"AI: {response}")

# Start chatting
chat()

--2025-02-12 22:24:10--  https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
Resolving www.cs.cornell.edu (www.cs.cornell.edu)... 132.236.207.53
Connecting to www.cs.cornell.edu (www.cs.cornell.edu)|132.236.207.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9916637 (9.5M) [application/zip]
Saving to: ‘cornell_movie_dialogs_corpus.zip.1’

cornell_movie_dialo 100%[===================>]   9.46M  7.98MB/s    in 1.2s    

2025-02-12 22:24:12 (7.98 MB/s) - ‘cornell_movie_dialogs_corpus.zip.1’ saved [9916637/9916637]

Archive:  cornell_movie_dialogs_corpus.zip
replace cornell movie-dialogs corpus/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace __MACOSX/cornell movie-dialogs corpus/._.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace cornell movie-dialogs corpus/chameleons.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace __MACOSX/cornell movie-dialogs corpus/._chameleons.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
rep

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter: